In [ ]:
# Supplementary Figures 8–14: synapse-to-tracheole distances

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

source_dir = Path('data/synapse_proximity')
out_dir = Path('supp_outputs')
out_dir.mkdir(parents=True, exist_ok=True)
seed = 20260806
nt_order = ['ACH', 'GABA', 'GLUT', 'DA', 'SER', 'OCT']
nt_colors = {'ACH': '#2020df', 'GABA': '#df2020', 'GLUT': '#df9c20',
             'DA': '#963939', 'OCT': '#107010', 'SER': '#701070'}
distance_col = 'distance_to_nearest_trachea_um'
superregions = {
    'OL': ['ME_L', 'ME_R', 'AME_L', 'AME_R', 'LO_L', 'LO_R', 'LOP_L', 'LOP_R'],
    'MB': ['MB_CA_R', 'MB_CA_L', 'MB_PED_L', 'MB_PED_R', 'MB_VL_R', 'MB_VL_L', 'MB_ML_R', 'MB_ML_L'],
    'CX': ['FB', 'EB', 'PB', 'NO'],
    'LX': ['BU_L', 'BU_R', 'LAL_R', 'LAL_L', 'GA_R', 'GA_L'],
    'VLNP': ['AOTU_R', 'AOTU_L', 'PVLP_L', 'AVLP_L', 'PVLP_R', 'AVLP_R', 'PLP_L', 'PLP_R', 'WED_L', 'WED_R'],
    'LH': ['LH_L', 'LH_R'],
    'SNP': ['SLP_L', 'SLP_R', 'SIP_L', 'SIP_R', 'SMP_R', 'SMP_L'],
    'INP': ['CRE_L', 'CRE_R', 'ICL_L', 'SCL_L', 'ICL_R', 'SCL_R', 'IB_L', 'IB_R', 'ATL_R', 'ATL_L'],
    'AL': ['AL_L', 'AL_R'],
    'VMNP': ['VES_L', 'VES_R', 'EPA_R', 'EPA_L', 'GOR_L', 'GOR_R', 'SPS_L', 'SPS_R', 'IPS_R', 'IPS_L'],
    'PENP': ['SAD', 'AMMC_R', 'AMMC_L', 'FLA_R', 'FLA_L', 'CAN_R', 'CAN_L', 'PRW'],
    'GNG': ['GNG'],
    'OCG': ['OCG'],
}

all_exact = pd.read_csv(source_dir / 'neuropil_distance_summary.csv')
nt_exact = pd.read_csv(source_dir / 'neuropil_nt_distance_summary.csv')
all_sample = pd.read_csv(source_dir / 'neuropil_violin_sample.csv.gz')
nt_sample = pd.read_csv(source_dir / 'neuropil_nt_violin_sample.csv.gz')
for table in (all_exact, nt_exact, all_sample, nt_sample):
    table['neuropil'] = table['neuropil'].astype('string').str.strip()
for name in ('LA_L', 'LA_R'):
    all_exact = all_exact[all_exact['neuropil'] != name]
    nt_exact = nt_exact[nt_exact['neuropil'] != name]
    all_sample = all_sample[all_sample['neuropil'] != name]
    nt_sample = nt_sample[nt_sample['neuropil'] != name]
nt_exact['nt_type'] = nt_exact['nt_type'].str.upper()
nt_sample['nt_type'] = nt_sample['nt_type'].str.upper()
region_map = {n: r for r, names in superregions.items() for n in names}

plt.rcParams.update({'font.family': 'Arial', 'font.size': 10, 'svg.fonttype': 'none',
                     'axes.linewidth': .7})

def sample_summary(df, columns):
    return df.groupby(columns)[distance_col].agg(
        sample_median_um='median', sample_q1_um=lambda x: x.quantile(.25),
        sample_q3_um=lambda x: x.quantile(.75)).reset_index()

def weighted_superregion_sample(sample, exact, columns, n, random_seed):
    exact_n = exact.groupby(columns)['n'].sum().rename('exact_n')
    available = sample.groupby(columns).size().rename('available_n')
    weighted = sample.join((exact_n / available).rename('weight'), on=columns)
    weighted['super_region'] = weighted['neuropil'].map(region_map)
    rng = np.random.default_rng(random_seed)
    blocks = []
    groups = ['super_region'] + (['nt_type'] if 'nt_type' in columns else [])
    for _, group in weighted.groupby(groups, sort=True):
        weights = group['weight'].to_numpy(float)
        indices = rng.choice(len(group), size=n, replace=True, p=weights / weights.sum())
        blocks.append(group.iloc[indices])
    return pd.concat(blocks, ignore_index=True)

def violin(ax, sample, column, names, color):
    values = [sample.loc[sample[column] == name, distance_col].dropna().to_numpy(float)
              for name in names]
    parts = ax.violinplot(values, positions=np.arange(1, len(names) + 1), vert=False,
                          widths=.78, showmedians=False, showextrema=False, bw_method='scott')
    for body in parts['bodies']:
        body.set_facecolor(color)
        body.set_edgecolor(color)
        body.set_alpha(.58)
    ax.scatter([np.median(v) for v in values], np.arange(1, len(names) + 1),
               s=25, c='#202020', edgecolors='white', linewidth=.3)
    ax.set_yticks(np.arange(1, len(names) + 1), names)
    ax.invert_yaxis()
    ax.set_xlim(0, np.quantile(np.concatenate(values), .995))
    ax.set_xlabel('Distance to nearest tracheole (µm)')
    ax.spines[['top', 'right']].set_visible(False)

def median_iqr(ax, summary, column, color):
    data = summary.sort_values(column).reset_index(drop=True)
    y = np.arange(len(data))
    mid = data['sample_median_um'].to_numpy(float)
    low = data['sample_q1_um'].to_numpy(float)
    high = data['sample_q3_um'].to_numpy(float)
    ax.errorbar(mid, y, xerr=[mid-low, high-mid], fmt='o', color=color,
                capsize=2, ms=3.5, lw=.8)
    ax.set_yticks(y, data[column].astype(str))
    ax.invert_yaxis()
    ax.set_xlabel('Median distance to nearest tracheole (µm)')
    ax.spines[['top', 'right']].set_visible(False)

def nt_heatmap(summary, column, ax, annotate=False):
    table = summary.pivot(index=column, columns='nt_type', values='sample_median_um')
    table = table.reindex(sorted(table.index), axis=0).reindex(nt_order, axis=1)
    matrix = table.to_numpy(float)
    image = ax.imshow(matrix, cmap='viridis', aspect='auto')
    ax.set_xticks(np.arange(len(nt_order)), nt_order)
    ax.set_yticks(np.arange(len(table)), table.index, fontsize=8 if len(table)>30 else 10)
    ax.tick_params(length=0)
    if annotate:
        midpoint = np.nanmedian(matrix)
        for (i, j), value in np.ndenumerate(matrix):
            if np.isfinite(value):
                ax.text(j, i, f'{value:.1f}', ha='center', va='center', fontsize=8,
                        color='white' if value <= midpoint else 'black')
    return image

# Supplementary Figure 8
regions = sorted(all_sample['neuropil'].dropna().unique())
for page, start in enumerate(range(0, len(regions), 20), 1):
    names = regions[start:start + 20]
    fig, ax = plt.subplots(figsize=(7, max(4, .34 * len(names) + 1)))
    violin(ax, all_sample, 'neuropil', names, '#3f6f8f')
    fig.tight_layout()
    fig.savefig(out_dir / f'supp_fig8_page{page:02d}.svg', bbox_inches='tight')
    plt.show()

# Supplementary Figure 9
summary = sample_summary(all_sample, ['neuropil'])
fig, ax = plt.subplots(figsize=(7, max(4, .27 * len(summary) + 1)))
median_iqr(ax, summary, 'neuropil', '#3f6f8f')
fig.tight_layout()
fig.savefig(out_dir / 'supp_fig9.svg', bbox_inches='tight')
plt.show()

# Supplementary Figure 10
sample = weighted_superregion_sample(all_sample, all_exact, ['neuropil'], 10000, seed + 1)
names = sorted(sample['super_region'].dropna().unique())
fig, ax = plt.subplots(figsize=(6.7, 5))
violin(ax, sample, 'super_region', names, '#3f6f8f')
fig.tight_layout()
fig.savefig(out_dir / 'supp_fig10a.svg', bbox_inches='tight')
plt.show()
summary = sample_summary(sample, ['super_region'])
fig, ax = plt.subplots(figsize=(6.7, 5))
median_iqr(ax, summary, 'super_region', '#3f6f8f')
fig.tight_layout()
fig.savefig(out_dir / 'supp_fig10b.svg', bbox_inches='tight')
plt.show()

# Supplementary Figure 11
for nt in nt_order:
    sample = nt_sample[nt_sample['nt_type'] == nt]
    regions = sorted(sample['neuropil'].dropna().unique())
    for page, start in enumerate(range(0, len(regions), 20), 1):
        names = regions[start:start + 20]
        fig, ax = plt.subplots(figsize=(7, max(4, .34 * len(names) + 1)))
        violin(ax, sample, 'neuropil', names, nt_colors[nt])
        fig.tight_layout()
        fig.savefig(out_dir / f'supp_fig11_{nt.lower()}_page{page:02d}.svg', bbox_inches='tight')
        plt.show()

# Supplementary Figure 12
sample = weighted_superregion_sample(nt_sample, nt_exact, ['neuropil', 'nt_type'], 5000, seed + 2)
fig, axes = plt.subplots(3, 2, figsize=(10.2, 11.2))
for ax, nt in zip(axes.ravel(), nt_order):
    subset = sample[sample['nt_type'] == nt]
    names = sorted(subset['super_region'].dropna().unique())
    violin(ax, subset, 'super_region', names, nt_colors[nt])
    ax.set_title(nt, loc='left')
fig.tight_layout()
fig.savefig(out_dir / 'supp_fig12.svg', bbox_inches='tight')
plt.show()

# Supplementary Figure 13
summary = sample_summary(nt_sample, ['neuropil', 'nt_type'])
fig, ax = plt.subplots(figsize=(6.7, max(5, .22 * summary['neuropil'].nunique() + 1.45)))
image = nt_heatmap(summary, 'neuropil', ax)
fig.colorbar(image, ax=ax, label='Median distance to nearest tracheole (µm)', fraction=.035, pad=.025)
fig.tight_layout()
fig.savefig(out_dir / 'supp_fig13.svg', bbox_inches='tight')
plt.show()

# Supplementary Figure 14
sample = weighted_superregion_sample(nt_sample, nt_exact, ['neuropil', 'nt_type'], 5000, seed + 2)
summary = sample_summary(sample, ['super_region', 'nt_type'])
fig, ax = plt.subplots(figsize=(6.7, 5))
image = nt_heatmap(summary, 'super_region', ax, annotate=True)
fig.colorbar(image, ax=ax, label='Median distance to nearest tracheole (µm)', fraction=.035, pad=.025)
fig.tight_layout()
fig.savefig(out_dir / 'supp_fig14.svg', bbox_inches='tight')
plt.show()